# EIS — Train models on real Kaggle data (India-only)

This is the **real-data** variant of `EIS_Train_Models.ipynb`, rewritten to use **only India-based datasets** so the learned pricing distribution matches what the Android app sees in production.

Fare estimator is trained on:

1. **Uber Ride Analytics Dashboard** ([`yashdevladdha/uber-ride-analytics-dashboard`](https://www.kaggle.com/datasets/yashdevladdha/uber-ride-analytics-dashboard)) — real Uber India bookings with native INR fares, ride distance, and vehicle type (Bike / eBike / Auto / Go Mini / Go Sedan / Premier Sedan / Uber XL). Covers all 4 tier classes.
2. **Synthetic CATALOG rows** for Ola + Rapido — neither publishes pricing data, so their rows are generated from the same Bengaluru CATALOG that `SmartRideDataRepository` prices with on-device.

Surge LSTM keeps the synthetic Bengaluru commute curve (no Indian 5-min surge history is public) but pulls **real Indian weather** (temperature / precip / wind / visibility) from:

3. **Indian Weather Repository — Daily Snapshot** ([`nelgiriyewithana/indian-weather-repository-daily-snapshot`](https://www.kaggle.com/datasets/nelgiriyewithana/indian-weather-repository-daily-snapshot)) — daily weather snapshots for Indian cities.

**Why India-only:** earlier revisions used Boston + NYC data with a USD→INR multiplier. That kept *units* in INR but the *pricing structure* (per-km rates, surge response, demand shape) was American, which biased the model toward the wrong rate card.

**Provider mapping:** Uber → UBER (real rows), Ola → OLA (synthetic), Rapido → RAPIDO (synthetic).

**No pre-trained weights** — both models initialise from random Glorot uniform / orthogonal kernels with a fixed seed (42).

**Currency:** all fares are already INR; no conversion applied.

Outputs (same four artefacts as the synthetic notebook):
```
surge_forecast.tflite
surge_forecast_meta.json
fare_estimator.tflite
fare_estimator_meta.json
```

Runtime: ~2–4 min on a Colab CPU (Indian datasets are much smaller than Boston).

## 1. Install dependencies

Colab already ships compatible TensorFlow / NumPy / pandas — pinning older versions causes a numpy ABI mismatch on import (`numpy.dtype size changed, may indicate binary incompatibility`) because pandas was compiled against the version Colab shipped. We only install **kagglehub**, which Colab doesn't have by default.

**If you already hit the ABI error from a previous run:** Colab → **Runtime → Restart session** first, then re-run from the top with the cell below.

In [ ]:
!pip install --quiet kagglehub

import tensorflow as tf, numpy as np, pandas as pd
print(f'tensorflow {tf.__version__}  numpy {np.__version__}  pandas {pd.__version__}')

## 2. Authenticate with Kaggle

**Path A — `kagglehub.login()`** (interactive, recommended):
    
Run the cell, click the link it prints, paste the returned token back into the prompt.

**Path B — upload `kaggle.json`** (if `login()` doesn't work in your Colab region):
    
Go to kaggle.com → Settings → API → "Create New Token" → save the downloaded `kaggle.json` and run the Path B cell below.

**Path C — skip Kaggle entirely:** download the two CSVs manually, upload them via `files.upload()`, then set `os.environ['UBER_INDIA_CSV']` and `os.environ['INDIA_WEATHER_CSV']` before running the trainer.

In [ ]:
# Path A — interactive login
import kagglehub
kagglehub.login()

In [ ]:
# Path B (alternative) — upload kaggle.json instead. Run this cell only if Path A failed.
# import os
# from google.colab import files
# uploaded = files.upload()  # pick kaggle.json
# os.makedirs('/root/.kaggle', exist_ok=True)
# !mv kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json

## 3. Download the Uber India rides dataset

Primary fare-training source — real Uber India bookings with native INR fares and all 4 tier classes. ~150k rows, ~50 MB download.

The trainer defensively resolves the column names (the dataset has been re-uploaded under slightly different schemas), so don't worry if the exact header spelling differs.

In [ ]:
import kagglehub, os, glob

uber_path = kagglehub.dataset_download('yashdevladdha/uber-ride-analytics-dashboard')
print('Uber India dataset dir:', uber_path)

uber_csvs = glob.glob(os.path.join(uber_path, '**', '*.csv'), recursive=True)
print('Uber India csvs:', uber_csvs)
assert uber_csvs, 'No CSV found inside the Uber India dataset download'
# Prefer the file whose name mentions bookings / rides / uber.
preferred_keys = ('booking', 'ride', 'uber')
uber_csv = next(
    (p for p in uber_csvs if any(k in os.path.basename(p).lower() for k in preferred_keys)),
    uber_csvs[0],
)
os.environ['UBER_INDIA_CSV'] = uber_csv
print('UBER_INDIA_CSV =', uber_csv)

## 4. Download the Indian weather dataset

Source of real weather features for the surge LSTM. ~30 MB download.

If this cell fails (e.g. dataset gated for your account), the trainer still runs — it falls back to a synthetic standard-normal weather pool and prints a warning.

In [ ]:
weather_path = kagglehub.dataset_download('nelgiriyewithana/indian-weather-repository-daily-snapshot')
print('Indian weather dataset dir:', weather_path)

weather_csvs = glob.glob(os.path.join(weather_path, '**', '*.csv'), recursive=True)
print('Weather csvs:', weather_csvs)
assert weather_csvs, 'No CSV found inside the weather dataset download'
weather_csv = next(
    (p for p in weather_csvs if 'weather' in os.path.basename(p).lower()),
    weather_csvs[0],
)
os.environ['INDIA_WEATHER_CSV'] = weather_csv
print('INDIA_WEATHER_CSV =', weather_csv)

## 5. Upload the trainer

Pick `train_on_real_data.py` from the repo's `ml_model/colab/` folder when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'train_on_real_data.py' in uploaded, 'Please upload train_on_real_data.py'

## 6. Train

Each model prints its architecture (LSTM / MLP layer-by-layer) and a banner confirming **no pre-trained weights are loaded** before training starts. Surge LSTM (~1 min, Bengaluru curve + real Indian weather) → fare MLP (~1–2 min, real Uber India + synthetic Ola/Rapido rows). Per-row predictions printed at the end of stage 2 are a quick eyeball check.

In [ ]:
%run train_on_real_data.py

## 7. Sanity-check the artefacts

Confirm all four files exist and inspect the meta JSONs. The fare meta carries a `training_data` block recording row counts (Uber India + synth) and the `output_order` / `quantiles` for the 3-head quantile regression. The surge meta carries a `weather_normalisation` block with per-feature means and stds so the Android side can z-score live readings the same way before feeding the model.

In [ ]:
import json, pathlib
for name in ['surge_forecast.tflite', 'surge_forecast_meta.json',
             'fare_estimator.tflite', 'fare_estimator_meta.json']:
    p = pathlib.Path(name)
    assert p.exists(), f'MISSING: {name}'
    print(f'{name:32s} {p.stat().st_size / 1024:>8.1f} KiB')

print('\nSurge meta:')
print(json.dumps(json.loads(pathlib.Path('surge_forecast_meta.json').read_text()), indent=2))
print('\nFare meta:')
print(json.dumps(json.loads(pathlib.Path('fare_estimator_meta.json').read_text()), indent=2))

## 8. Download to your machine

Save each file straight into `app/src/main/assets/` in the Android project, then rebuild.

In [ ]:
from google.colab import files
for name in ['surge_forecast.tflite', 'surge_forecast_meta.json',
             'fare_estimator.tflite', 'fare_estimator_meta.json']:
    files.download(name)